In [ ]:
import os
import uuid
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from datetime import datetime
from langgraph.checkpoint.memory import InMemorySaver
from psycopg import AsyncConnection
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from urllib.parse import quote_plus


# LangChain & MCP imports
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

POSTGRESQL_DB_USER = os.getenv("POSTGRESQL_DB_USER")
POSTGRESQL_DB_PASSWORD = os.getenv("POSTGRESQL_DB_PASSWORD")
POSTGRESQL_DB_HOST = os.getenv("POSTGRESQL_DB_HOST")
POSTGRESQL_DB_PORT = os.getenv("POSTGRESQL_DB_PORT", "5432")
POSTGRESQL_DB_NAME = os.getenv("POSTGRESQL_DB_NAME")

# Validate required fields
if not all(
    [
        POSTGRESQL_DB_USER,
        POSTGRESQL_DB_PASSWORD,
        POSTGRESQL_DB_HOST,
        POSTGRESQL_DB_PORT,
        POSTGRESQL_DB_NAME,
    ]
):
    raise EnvironmentError(
        "Missing one or more required environment variables for PostgreSQL: "
        "DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_NAME"
    )

# URL-encode password for safe use in connection string
pg_password_encoded = quote_plus(POSTGRESQL_DB_PASSWORD)
POSTGRESQL_DATABASE_URL = (
    f"postgresql://{POSTGRESQL_DB_USER}:{pg_password_encoded}@"
    f"{POSTGRESQL_DB_HOST}:{POSTGRESQL_DB_PORT}/{POSTGRESQL_DB_NAME}"
)


async def get_async_checkpointer(use_postgres: bool = True):
    if not use_postgres:
        return InMemorySaver()
    # conn = Connection.connect(conninfo=POSTGRESQL_DATABASE_URL, autocommit=True)
    conn = await AsyncConnection.connect(
        conninfo=POSTGRESQL_DATABASE_URL, autocommit=True
    )
    checkpointer = AsyncPostgresSaver(conn=conn)
    await checkpointer.setup()
    return checkpointer


SANDBOX_DIR = "./sandbox"

# MCP Configuration
MAIN_LLM_BASE_URL = os.getenv("MAIN_LLM_BASE_URL")
MAIN_LLM_API_KEY = os.getenv("MAIN_LLM_API_KEY")
MAIN_LLM_MODEL_NAME = os.getenv("MAIN_LLM_MODEL_NAME")


mcp_client = MultiServerMCPClient(
    connections={
        "mcp-manager": {
            "transport": "sse",
            "url": "http://host.docker.internal:8000/sse",
        },
    }
)

config: RunnableConfig = {"configurable": {"thread_id": str(uuid.uuid4())}}


tools = await mcp_client.get_tools()


system_prompt = f"""
You are a cybersecurity analyst with MCP tool access.
Current date: {datetime.now().strftime("%Y-%m-%d")}

Use MCP tools to gather information and perform calculations.
Always explain your tool usage clearly in the response.
"""


model = ChatOpenAI(
    base_url=MAIN_LLM_BASE_URL,
    api_key=MAIN_LLM_API_KEY,
    model=MAIN_LLM_MODEL_NAME,
)
USE_POSTGRES = True
checkpointer = await get_async_checkpointer(use_postgres=USE_POSTGRES)


agent = create_deep_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=checkpointer,
    backend=FilesystemBackend(root_dir="./sandbox", virtual_mode=True),
)

In [ ]:
prompt = "https://zhuanlan.zhihu.com/p/1994423521271637277 这个说的什么"
async for res in agent.astream(
    input={"messages": [HumanMessage(content=prompt)]},
    config=config,
    stream_mode="values"
):
    print(res)